In [2]:
import modified_didppy as m_dp
import vrplib
import numpy as np
import math
import tempfile
import os
from scipy.spatial.distance import cdist
import pulp
import re

# **Data**

# **TSP model**

In [6]:
# Sets
n_nodes = n_nodes      # n (Total nodes including depot)
n_customers = n_nodes - 1             # Customers only
n_vehicles = num_vehicles                  # p (As per filename k7)

# Indices
# Nodes: 0 is depot, 1..52 are customers
nodes = range(n_nodes)                # {0, ..., n}
customers = range(1, n_nodes)         # {1, ..., n}
vehicles = range(n_vehicles)          # {0, ..., p-1} (Python 0-indexed)
c = dist_matrix                  # Cost matrix

#Model declaration
model = pulp.LpProblem("CVRP_LP_Relaxation", pulp.LpMinimize)

# --- VARIABLES ---

# X: Flow variables (Relaxed to Continuous 0 to 1)
x = pulp.LpVariable.dicts(
    "x", 
    ((r, i, j) for r in vehicles for i in nodes for j in nodes if i != j), 
    cat='Continuous', lowBound=0, upBound=1
)

# U: MTZ Load variables (MISSING IN YOUR SNIPPET)
# Even in relaxation, these help tighten the bound
u = pulp.LpVariable.dicts(
    "u", 
    customers, 
    lowBound=0, 
    upBound=capacity, 
    cat='Continuous'
)

# --- OBJECTIVE ---
model += pulp.lpSum(
    c[i][j] * x[r, i, j] 
    for r in vehicles 
    for i in nodes 
    for j in nodes 
    if i != j
), "Minimize_Total_Cost"

# --- CONSTRAINTS ---

# (2) Each customer visited exactly once (Sum of fractions = 1.0)
for j in customers:
    model += pulp.lpSum(x[r, i, j] for r in vehicles for i in nodes if i != j) == 1, f"Visit_{j}"
    
# (3) Each vehicle leaves depot exactly once
for r in vehicles:
    model += pulp.lpSum(x[r, 0, j] for j in customers) == 1, f"Leave_Depot_{r}"

# (4) Flow Conservation
for j in nodes: 
    for r in vehicles:
        inflow = pulp.lpSum(x[r, i, j] for i in nodes if i != j)
        outflow = pulp.lpSum(x[r, j, k] for k in nodes if j != k)
        model += inflow == outflow, f"Flow_Balance_{j}_{r}"

# (5) Capacity Constraint
for r in vehicles:
    model += pulp.lpSum(demands[j] * x[r, i, j] for i in nodes for j in customers if i != j) <= capacity, f"Cap_{r}"

# (6) MTZ Subtour Elimination (Based on provided Image)
# ------------------------------------------------------------------
# The image defines the constraint (1) as: 
# u_j - u_i >= q_j - Q * (1 - x_ijk)
#
# Logic from image:
# IF vehicle r drives i -> j (x=1): 
#    Constraint becomes: u_j >= u_i + q_j 
#    This ensures u_j is at least q_j (demand at j) more than u_i.
#
# IF vehicle r does NOT drive i -> j (x=0):
#    Constraint becomes: u_j - q_j >= u_i - Q
#    This is always valid because u_j >= q_j and u_i <= Q.
# ------------------------------------------------------------------

for r in vehicles:
    for i in customers:
        for j in customers:
            if i != j:
                # We implement: u_j - u_i >= q_j - Q * (1 - x_ijk)
                # Note: We use the parentheses (1 - x[...]) for clarity matching the image
                model += u[j] - u[i] >= demands[j] - capacity * (1 - x[r, i, j]), f"MTZ_{r}_{i}_{j}"

# Constraint (2) from Image: q_i <= u_i <= Q
# The image states: "q_j is the lowest possible value of u_j and Q is the greatest"
for i in customers:
    model += u[i] >= demands[i]  # Lower bound (q_i)
    model += u[i] <= capacity    # Upper bound (Q)

# ==========================================
# SOLVE (LP RELAXATION)
# ==========================================
# Since this is LP, it solves extremely fast compared to MIP
solver = pulp.CPLEX_CMD(timeLimit=60, msg=True) 

try:
    model.solve(solver)
except Exception as e:
    print(f"CPLEX not found, using default: {e}")
    model.solve()

# ==========================================
# 4. RESULTS
# ==========================================
print(f"\nStatus: {pulp.LpStatus[model.status]}")
print(f"LOWER BOUND (LP Relaxation Cost): {pulp.value(model.objective)}")

# Visualizing Fractional Flows
print("\nSignificant Fractional Flows (> 0):")
print("(Format: Vehicle | From -> To | Flow Amount)")
print("-" * 40)

count = 0
for r in vehicles:
    for i in nodes:
        for j in nodes:
            if i != j:
                val = pulp.value(x[r, i, j])
                # Check for non-zero flow (values like 0.5, 0.33 etc)
                if val and val > 0: 
                    print(f"Veh {r} | {i:2d} -> {j:2d} | {val:.2f}")
                    count += 1
                    if count > 20: # Limit output to avoid spamming
                        print("... (too many fractional edges to list)")
                        break
    if count > 20: break


Status: Optimal
LOWER BOUND (LP Relaxation Cost): 538.9978600507287

Significant Fractional Flows (> 0):
(Format: Vehicle | From -> To | Flow Amount)
----------------------------------------
Veh 0 |  0 -> 26 | 0.16
Veh 0 |  0 -> 27 | 0.84
Veh 0 |  4 -> 11 | 0.19
Veh 0 |  5 -> 20 | 0.08
Veh 0 |  6 -> 23 | 0.12
Veh 0 |  7 -> 13 | 0.16
Veh 0 |  8 -> 18 | 0.85
Veh 0 |  9 -> 22 | 0.16
Veh 0 | 11 ->  4 | 0.19
Veh 0 | 13 ->  7 | 0.16
Veh 0 | 18 -> 28 | 0.85
Veh 0 | 20 ->  5 | 0.08
Veh 0 | 21 -> 31 | 0.12
Veh 0 | 22 ->  9 | 0.16
Veh 0 | 23 ->  6 | 0.12
Veh 0 | 26 ->  0 | 0.16
Veh 0 | 27 ->  0 | 0.84
Veh 0 | 28 ->  8 | 0.85
Veh 0 | 31 -> 21 | 0.12
Veh 1 |  0 -> 14 | 0.84
Veh 1 |  0 -> 27 | 0.16
... (too many fractional edges to list)
Veh 1 |  1 -> 12 | 0.21
... (too many fractional edges to list)
Veh 1 |  2 ->  3 | 0.21
... (too many fractional edges to list)
Veh 1 |  3 ->  2 | 0.21
... (too many fractional edges to list)
Veh 1 |  8 -> 18 | 0.15
... (too many fractional edges to list)
Veh 1 | 